In [0]:
%run ./_config

In [0]:
# simple pipeline health report
health = spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {SILVER_SCHEMA}.fct_sales) AS fact_rows,
        (SELECT COUNT(DISTINCT store_id)  FROM {SILVER_SCHEMA}.dim_stores  WHERE is_current = TRUE) AS active_stores,
        (SELECT COUNT(DISTINCT item_id)   FROM {SILVER_SCHEMA}.dim_products WHERE is_current = TRUE) AS active_products,
        (SELECT COUNT(DISTINCT vendor_id) FROM {SILVER_SCHEMA}.dim_vendors  WHERE is_current = TRUE) AS active_vendors,
        (SELECT SUM(CASE WHEN store_sk IS NULL THEN 1 ELSE 0 END) FROM {SILVER_SCHEMA}.fct_sales) AS null_store_fks,
        (SELECT MIN(sale_date) FROM {SILVER_SCHEMA}.fct_sales) AS earliest_sale,
        (SELECT MAX(sale_date) FROM {SILVER_SCHEMA}.fct_sales) AS latest_sale
""")
health.display()

In [0]:
# read upstream task values (works only within Workflow)
try:
    rows_ingested = dbutils.jobs.taskValues.get(
        taskKey="ingest_api",
        key="rows_ingested",
        default=0
    )
    load_date = dbutils.jobs.taskValues.get(
        taskKey="ingest_api",
        key="load_date",
        default="unknown"
    )
    print(f"Ingestion reported: {rows_ingested:,} rows, load date: {load_date}")
except Exception:
    print("Task values unavailable")

In [0]:
# gold layer validation
gold_tables = [
    "gold_monthly_sales", "gold_dow_patterns", "gold_county_performance",
    "gold_city_rankings", "gold_vendor_scorecard",
    "gold_category_market_share", "gold_store_rankings"
]

for tbl in gold_tables:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_SCHEMA}.{tbl}").collect()[0]["cnt"]
    status = "OK" if count > 0 else "Empty, check pipeline"
    print(f"  {tbl}: {count:,} rows — {status}")